# Introduction to PyTorch - Exercises
## Computer Vision and Image Processing - Lab Session 5
### Prof: Luigi Di Stefano, luigi.distefano@unibo.it
### Tutor: Iacopo Curti, iacopo.curti2@unibo.it - Matteo Fusconi, matteo.fusconi6@unibo.it


# 1: Backpropagation

Consider the function:
$$y = \frac{1}{N}\sum_i^N \sin(x_i) \cdot e^{x_i}$$

1. Calculate the gradient manually with tensor operations, namely the values of the vector $\frac{\partial y}{\partial x}$
2. Calculate the gradient with autograd to verify its correctness

Remember that you can split the function y(x) into:
* $a = \sin(x)$
* $b = e^x$
* $c = a \cdot b$
* $y = \text{mean}(c) = \frac{1}{N}\sum_i^N c_i$

and that the final gradient can be computed using the product rule and the chain rule as $\frac{\partial y}{\partial x} = \frac{\partial y}{\partial c}\left(\frac{\partial c}{\partial a}\frac{\partial a}{\partial x} + \frac{\partial c}{\partial b}\frac{\partial b}{\partial x}\right)$

In [ ]:
import torch

N = 5
x = torch.arange(N, dtype=torch.float32)

a = torch.sin(x)          # a = sin(x)
b = torch.exp(x)          # b = exp(x)
c = a * b                 # c = a * b = sin(x) * exp(x)
y = torch.mean(c)         # y = mean(c)

# Backward pass, one gradient at a time (chain rule, node by node)
dy_dc = torch.full_like(c, 1 / N)   # dy/dc_i = 1/n  (from the mean)
dc_da = b                           # dc/da = b      (from c = a * b)
dc_db = a                           # dc/db = a      (from c = a * b)
da_dx = torch.cos(x)          # da/dx = cos(x) (from a = sin(x))
db_dx = torch.exp(x)          # db/dx = exp(x) (from b = exp(x))

grad = dy_dc * (dc_da * da_dx + dc_db * db_dx)

print(grad)

In [ ]:
N = 5
x = torch.arange(N, dtype=torch.float32, requires_grad=True)

y = torch.mean(torch.sin(x) * torch.exp(x))
y.backward()

print(x.grad)

# 2. Improve the classification accuracy

Try to achieve a better accuracy on the validation set by modifying the previous model and manually tuning the hyper-parameters such as the number of epochs, the learning rate, and the number of hidden layers.


Also, add [Batch Normalization](https://arxiv.org/abs/1502.03167). You can find the normalization layers already implemented in PyTorch [here](https://pytorch.org/docs/stable/nn.html#normalization-layers). Use `torch.nn.Sequential` to stack together a block with the following
structure: FC $\rightarrow$ BatchNorm $\rightarrow$ Activation.

You should easily get 80% accuracy on the validation set.


In [ ]:
import pandas as pd
import seaborn as sn

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.notebook import tqdm

from torch.optim import Adam
from torch.utils.data import random_split, DataLoader
from torchvision import transforms as T, datasets

In [ ]:
tsfms = T.Compose([
    T.ToTensor(),
    T.Lambda(lambda x: x.flatten())
])
train_dset = datasets.SVHN(
    root=".",
    split="train",
    transform=tsfms,
    download=True
)
test_dset = datasets.SVHN(
    root=".",
    split="test",
    transform=tsfms,
    download=True
)
n_classes = 10
input_dim = len(train_dset[0][0])

split_dsets = random_split(
    train_dset,
    [len(train_dset) - 5000, 5000]
)
train_subdset = split_dsets[0]
val_subdset = split_dsets[1]

batch_size = 256

train_dl = DataLoader(
    train_subdset,
    batch_size=batch_size,
    shuffle=True
)
val_dl = DataLoader(
    val_subdset,
    batch_size=batch_size
)
test_dl = DataLoader(
    test_dset,
    batch_size=batch_size
)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def get_linear_bn_relu(input_dim, out_dim):
    """Return a Sequential module with a sandwich of Linear, Batch Norm, and ReLU.

    Args:
        input_dim: the number of input features.
        out_dim: the number of the output features.

    Returns: the created Sequential block.
    """
    modules = []
    modules.append(nn.Linear(input_dim, out_dim))
    modules.append(nn.BatchNorm1d(out_dim))
    modules.append(nn.ReLU())

    return nn.Sequential(*modules)

class SimpleClassifierBN(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_classes, n_hidden_layers=0):
        super().__init__()

        self.first = get_linear_bn_relu(input_dim, hidden_dim)
        self.last = nn.Linear(hidden_dim, n_classes)

        self.hidden_layers = nn.ModuleList([
            get_linear_bn_relu(hidden_dim, hidden_dim) for i in range(n_hidden_layers)
        ])

    def forward(self, x):
        x = self.first(x)
        for layer in self.hidden_layers:
            x = layer(x)
        x = self.last(x)

        return x

In [ ]:
def ncorrect(scores, y):
    y_hat = torch.argmax(scores, -1)
    return (y_hat == y).sum()

def accuracy(scores, y):
    correct = ncorrect(scores, y)
    return correct.true_divide(y.shape[0])

def train_loop(model, train_dl, epochs, opt, val_dl=None, verbose=False):
    best_val_acc = 0
    best_params = []
    best_epoch = -1

    for e in tqdm(range(epochs)):
        model.train()
        # Train
        train_loss = 0
        train_samples = 0
        train_acc = 0
        for train_data in train_dl:
            imgs = train_data[0].to(device)
            labels = train_data[1].to(device)
            scores = model(imgs)
            loss = F.cross_entropy(scores, labels)
            train_loss += loss.item() * imgs.shape[0]
            train_samples += imgs.shape[0]
            train_acc += ncorrect(scores, labels).item()

            opt.zero_grad()  # clear
            loss.backward()  # fill
            opt.step()       # use

        train_acc /= train_samples
        train_loss /= train_samples

        # Validation
        model.eval()
        with torch.no_grad():
            val_loss = 0
            val_samples = 0
            val_acc = 0
            if val_dl is not None:
                for val_data in val_dl:
                    imgs = val_data[0].to(device)
                    labels = val_data[1].to(device)
                    val_scores = model(imgs)
                    val_loss += F.cross_entropy(val_scores, labels).item() * imgs.shape[0]
                    val_samples += imgs.shape[0]
                    val_acc += ncorrect(val_scores, labels).item()
                val_acc /= val_samples
                val_loss /= val_samples

            if val_dl is None or val_acc > best_val_acc:
                best_val_acc = val_acc if val_dl is not None else 0
                best_params = model.state_dict()
                torch.save(best_params, "best_model.pth")
                best_epoch = e

        if verbose and e % 5 == 0:
            print(f"Epoch {e}: train loss {train_loss:.3f} - train acc {train_acc:.3f}" + ("" if val_dl is None else f" - valid loss {val_loss:.3f} - valid acc {val_acc:.3f}"))

    if verbose and val_dl is not None:
        print(f"Best epoch {best_epoch}, best acc {best_val_acc}")

    return best_val_acc, best_params, best_epoch

In [ ]:
hidden_dim = 128
new_model = model = SimpleClassifierBN(
    input_dim,
    hidden_dim,
    n_classes,
    n_hidden_layers=3
)
new_model.to(device)
optimizer = torch.optim.Adam(new_model.parameters(), lr=0.01)
epochs = 25
best_val_acc, best_params, best_epoch = train_loop(
    model,
    train_dl,
    epochs,
    optimizer,
    val_dl,
    verbose=True)

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))

## Exercise 3: Confusion Matrix
Compute the confusion matrix for the test set. Then, from the confusion matrix, compute the accuracy score.

> **Note:** a confusion matrix $\mathbf{C}$ is a square matrix of shape `(num_classes, num_classes)`, where $\mathbf{C}_{ij}$ is the number of samples whose **true** class is $i$ and whose **predicted** class is $j$. Rows represent the true class, columns represent the predicted class.
>
> |                  | Pred: 0 | Pred: 1 | Pred: 2 |
> |------------------|:-------:|:-------:|:-------:|
> | **True: 0**      |   $C_{00}$   |   $C_{01}$   |   $C_{02}$   |
> | **True: 1**      |   $C_{10}$   |   $C_{11}$   |   $C_{12}$   |
> | **True: 2**      |   $C_{20}$   |   $C_{21}$   |   $C_{22}$   |
>
> The **diagonal** ($C_{ii}$) counts correct predictions; every off-diagonal entry is a specific kind of mistake (e.g. $C_{12}$ = samples truly of class 1 that got predicted as class 2).

In [ ]:
model.eval()
confusion_matrix = torch.zeros(n_classes, n_classes, dtype=torch.int)

with torch.no_grad():
    for test_data in test_dl:
        imgs, labels = test_data
        imgs = imgs.to(device)
        labels = labels.to(device)

        scores = new_model(imgs)
        preds = torch.argmax(scores, -1)
        for lbl, pred in zip(labels.view(-1), preds.view(-1)):
                confusion_matrix[lbl.int(), pred.int()] += 1

test_acc = (torch.diagonal(confusion_matrix).sum() / confusion_matrix.sum()).item()

print(test_acc)

In [ ]:
conf_mtrx_df = pd.DataFrame(confusion_matrix)
sn.heatmap(conf_mtrx_df, annot=True, fmt="d");